# Day 2: Automation Design

## Module 5: Automation in AI | Al Jazira Bank

In this notebook you will work with both the workflow inventory and the automation candidates datasets to design, evaluate, and plan automation proposals.

### What you will do
1. Join the workflow inventory with automation candidates data.
2. Analyse estimated savings against implementation effort and risk.
3. Build a prioritisation matrix for automation decisions.
4. Prepare data-backed inputs for your Lab D, E, and F deliverables.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load both datasets
workflows = pd.read_csv("../data/workflow_inventory.csv")
candidates = pd.read_csv("../data/automation_candidates.csv")

# Join on workflow_id
combined = workflows.merge(candidates, on="workflow_id", how="left")

print(f"Workflows with automation candidates: {candidates['workflow_id'].nunique()}")
print(f"Workflows without candidates: {combined['candidate_id'].isna().sum()}")
print()
combined.head(10)

In [ ]:
# Analyse automation candidates by type and effort
auto_only = combined.dropna(subset=["candidate_id"]).copy()

print("Automation type distribution:")
print(auto_only["automation_type"].value_counts())
print()

print("Implementation effort distribution:")
print(auto_only["implementation_effort"].value_counts())
print()

print("Risk level distribution:")
print(auto_only["risk_level"].value_counts())
print()

print("Requires human review:")
print(auto_only["requires_human_review"].value_counts())

In [ ]:
# Exercise: Build a prioritisation matrix
#
# Task 1: Map implementation_effort to numeric values (Low=1, Medium=2, High=3).
# Task 2: Calculate an ROI proxy = estimated_saving_hours / effort_score.
# Task 3: Create a 2x2 matrix: high savings + low effort (quick wins) vs others.
# Task 4: Visualise the matrix.

effort_map = {"Low": 1, "Medium": 2, "High": 3}
auto_only["effort_score"] = auto_only["implementation_effort"].map(effort_map)
auto_only["roi_proxy"] = auto_only["estimated_saving_hours"] / auto_only["effort_score"]

savings_median = auto_only["estimated_saving_hours"].median()
effort_median = auto_only["effort_score"].median()

def classify_quadrant(row):
    high_savings = row["estimated_saving_hours"] >= savings_median
    low_effort = row["effort_score"] <= effort_median
    if high_savings and low_effort:
        return "Quick Win"
    elif high_savings and not low_effort:
        return "Strategic Bet"
    elif not high_savings and low_effort:
        return "Maybe Later"
    else:
        return "Avoid"

auto_only["quadrant"] = auto_only.apply(classify_quadrant, axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
quad_colours = {
    "Quick Win": "#2ecc71",
    "Strategic Bet": "#3498db",
    "Maybe Later": "#f39c12",
    "Avoid": "#e74c3c",
}
for quadrant, group in auto_only.groupby("quadrant"):
    ax.scatter(
        group["effort_score"],
        group["estimated_saving_hours"],
        label=quadrant,
        c=quad_colours.get(quadrant, "grey"),
        s=100,
        alpha=0.8,
    )
    for _, row in group.iterrows():
        ax.annotate(
            row["process_name"][:20],
            (row["effort_score"], row["estimated_saving_hours"]),
            fontsize=7,
            ha="left",
            va="bottom",
        )

ax.axhline(y=savings_median, color="grey", linestyle="--", alpha=0.5)
ax.axvline(x=effort_median, color="grey", linestyle="--", alpha=0.5)
ax.set_xlabel("Implementation Effort (1=Low, 2=Medium, 3=High)")
ax.set_ylabel("Estimated Saving (hours/week)")
ax.set_title("Automation Prioritisation Matrix")
ax.legend(title="Quadrant")
plt.tight_layout()
plt.show()

In [ ]:
# Exercise: Build your final recommendation table
#
# Produce a table of your top 5 recommendations sorted by ROI proxy.
# Include: process name, department, automation type, estimated savings,
#          effort, risk, human review required, quadrant.
# Add a "justification" column with your one-sentence reasoning.

top5 = auto_only.sort_values("roi_proxy", ascending=False).head(5)

recommendation = top5[[
    "process_name", "department", "automation_type",
    "estimated_saving_hours", "implementation_effort",
    "risk_level", "requires_human_review", "quadrant", "roi_proxy"
]].copy()

print("Top 5 Automation Recommendations:")
print(recommendation.to_string(index=False))

# Reflection:
# - Do your top 5 match what you expected from the Day 1 analysis?
# - Are any "Quick Win" candidates also high-risk? What does that tell you?
# - Which recommendation would you present to leadership first, and why?
# - What data is missing that would make your recommendation stronger?